# A-S00 Standard SegResNet baseline

This notebook controls the A-S00 baseline through repository
source files. Training logic is not duplicated inside the notebook.

The baseline must remain standard MONAI SegResNet. Attention or
decoder architecture changes require a different experiment ID.

Stages:

1. Inspect contract and source
2. Verify shared GPU is idle
3. Run the full-patch batch-1 capacity smoke
4. Start patch training with `--stage train`
5. Monitor durable progress/checkpoints
6. After training, run final validation once with `--stage final`


In [ ]:
from pathlib import Path
from IPython.display import Code, display
import json
import os
import subprocess
import sys

REPO_ROOT = Path("/home/jupyter-mirae8/Aneu_MRA/model")
DATA_ROOT = Path("/home/jupyter-mirae8/Aneu_MRA/data")
RUN_ROOT = Path("/home/jupyter-mirae8/Aneu_MRA/runs")
VENV_PYTHON = REPO_ROOT / ".venv" / "bin" / "python"

os.environ["BRAINON_DATA_ROOT"] = str(DATA_ROOT)
os.environ["BRAINON_RUN_ROOT"] = str(RUN_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

print("Repository :", REPO_ROOT)
print("Data root  :", DATA_ROOT)
print("Run root   :", RUN_ROOT)
print("Venv Python:", VENV_PYTHON)


In [ ]:
FILES_TO_INSPECT = [
    "configs/experiments/aneurysm/A-S00.yaml",
    "src/brainon_ai/models/aneurysm/segresnet/network.py",
    "src/brainon_ai/models/aneurysm/segresnet/losses.py",
    "src/brainon_ai/models/aneurysm/segresnet/data.py",
    "src/brainon_ai/models/aneurysm/segresnet/train.py",
    "src/brainon_ai/models/aneurysm/segresnet/evaluate.py",
]

for relative_path in FILES_TO_INSPECT:
    path = REPO_ROOT / relative_path
    print("\nFILE:", relative_path)
    display(Code(filename=str(path), language="python"))


In [ ]:
gpu_processes = subprocess.run(
    [
        "nvidia-smi",
        "--query-compute-apps=pid,process_name,used_gpu_memory",
        "--format=csv,noheader,nounits",
    ],
    text=True,
    capture_output=True,
    check=True,
).stdout.strip()

print(gpu_processes or "PASS: GPU compute process 없음")


## Training launch

Do not change `START_TRAINING` until:

- repository tests pass;
- batch-1 full-patch GPU smoke passes;
- reviewed files are committed and pushed to `dev`;
- the shared GPU has no other compute processes.

Training runs as a separate process. Browser disconnection does not
normally stop it. If the process or server stops, rerunning the same
stage resumes from `checkpoint_latest.pth`.


In [ ]:
START_TRAINING = False

if START_TRAINING:
    LOG_ROOT = Path("/home/jupyter-mirae8/Aneu_MRA/process_logs")
    LOG_ROOT.mkdir(parents=True, exist_ok=True)

    log_path = LOG_ROOT / "A-S00_train.log"
    pid_path = LOG_ROOT / "A-S00_train.pid"

    environment = os.environ.copy()
    environment["PYTHONPATH"] = str(REPO_ROOT / "src")
    environment["BRAINON_DATA_ROOT"] = str(DATA_ROOT)
    environment["BRAINON_RUN_ROOT"] = str(RUN_ROOT)
    environment["PYTHONUNBUFFERED"] = "1"
    environment["CUDA_VISIBLE_DEVICES"] = "0"

    log_handle = log_path.open("a", encoding="utf-8")

    process = subprocess.Popen(
        [
            str(VENV_PYTHON),
            str(REPO_ROOT / "scripts/training/run_a_s00.py"),
            "--stage",
            "train",
        ],
        cwd=REPO_ROOT,
        env=environment,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    log_handle.close()
    pid_path.write_text(str(process.pid), encoding="utf-8")

    print("Training PID:", process.pid)
    print("Training log:", log_path)
else:
    print("LOCKED: START_TRAINING=False")


In [ ]:
progress_path = RUN_ROOT / "aneurysm" / "A-S00" / "progress.json"

if progress_path.is_file():
    print(json.dumps(
        json.loads(progress_path.read_text(encoding="utf-8")),
        indent=2,
        ensure_ascii=False,
    ))
else:
    print("No production progress.json yet")


## Final full-volume validation

Run only after patch training produces `checkpoint_final.pth`.
It uses `checkpoint_best.pth` and processes 46 validation patients
once. Patient-level resume state is saved after every completed case.


In [ ]:
START_FINAL_VALIDATION = False

if START_FINAL_VALIDATION:
    subprocess.run(
        [
            str(VENV_PYTHON),
            str(REPO_ROOT / "scripts/training/run_a_s00.py"),
            "--stage",
            "final",
        ],
        cwd=REPO_ROOT,
        env={
            **os.environ,
            "PYTHONPATH": str(REPO_ROOT / "src"),
            "BRAINON_DATA_ROOT": str(DATA_ROOT),
            "BRAINON_RUN_ROOT": str(RUN_ROOT),
            "CUDA_VISIBLE_DEVICES": "0",
        },
        check=True,
    )
else:
    print("LOCKED: START_FINAL_VALIDATION=False")
